# Initialize Dataset

In [4]:
# ============================================================
# CELL 1 — PQD-16c DATA INITIALIZATION
# ============================================================

import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader


# -------------------- CONFIG --------------------

SEED = 42
BATCH_SIZE = 64
INPUT_LENGTH = 640

DATA_DIR = Path("../dataset/PQD-16c-Data")

TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV   = DATA_DIR / "validation.csv"
TEST_CSV  = DATA_DIR / "test.csv"


# -------------------- REPRODUCIBILITY --------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# -------------------- DEVICE --------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("PQD-16c DATA INITIALIZATION")
print("=" * 60)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# -------------------- LOAD CSVs --------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

LABEL_COLUMN = train_df.columns[0]
SIGNAL_COLUMNS = train_df.columns[1:]

assert val_df.columns.tolist() == train_df.columns.tolist()
assert test_df.columns.tolist() == train_df.columns.tolist()
assert len(SIGNAL_COLUMNS) == INPUT_LENGTH


# -------------------- EXTRACT DATA --------------------

X_train = train_df[SIGNAL_COLUMNS].values.astype(np.float32)
X_val   = val_df[SIGNAL_COLUMNS].values.astype(np.float32)
X_test  = test_df[SIGNAL_COLUMNS].values.astype(np.float32)

y_train_raw = train_df[LABEL_COLUMN].values
y_val_raw   = val_df[LABEL_COLUMN].values
y_test_raw  = test_df[LABEL_COLUMN].values


# -------------------- LABEL ENCODING --------------------

class_names = sorted(np.unique(y_train_raw))
NUM_CLASSES = len(class_names)

label_to_index = {
    label: i for i, label in enumerate(class_names)
}

y_train = np.array(
    [label_to_index[x] for x in y_train_raw],
    dtype=np.int64
)

y_val = np.array(
    [label_to_index[x] for x in y_val_raw],
    dtype=np.int64
)

y_test = np.array(
    [label_to_index[x] for x in y_test_raw],
    dtype=np.int64
)

assert NUM_CLASSES == 16


# -------------------- NORMALIZATION --------------------
# IMPORTANT: only training statistics are used.

train_mean = X_train.mean()
train_std = X_train.std()

X_train = (X_train - train_mean) / train_std
X_val   = (X_val - train_mean) / train_std
X_test  = (X_test - train_mean) / train_std


# -------------------- TENSOR CONVERSION --------------------
# [N, 640] -> [N, 1, 640]

X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val_t   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t   = torch.tensor(y_val, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)


# -------------------- DATASETS --------------------

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset   = TensorDataset(X_val_t, y_val_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)


# -------------------- DATALOADERS --------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# -------------------- SUMMARY --------------------

print("\nClasses:", NUM_CLASSES)
print("Class names:", class_names)

print("\nShapes:")
print("X_train:", X_train_t.shape)
print("X_val  :", X_val_t.shape)
print("X_test :", X_test_t.shape)

print("\nLabels:")
print("y_train:", y_train_t.shape)
print("y_val  :", y_val_t.shape)
print("y_test :", y_test_t.shape)

print("\nNormalization:")
print(f"Train mean: {train_mean:.8f}")
print(f"Train std : {train_std:.8f}")

print("\n" + "=" * 60)
print("DATA READY")
print("=" * 60)

PQD-16c DATA INITIALIZATION
Device: cpu

Classes: 16
Class names: ['Flicker', 'Flicker+Harmonics', 'Flicker+Sag', 'Flicker+Swell', 'Harmonics', 'Impulsive Transient', 'Interruption', 'Interruption+Harmonics', 'Normal', 'Notch', 'Oscillatory transient', 'Sag', 'Sag+Harmonics', 'Spike', 'Swell', 'Swell+Harmonics']

Shapes:
X_train: torch.Size([1024, 1, 640])
X_val  : torch.Size([256, 1, 640])
X_test : torch.Size([320, 1, 640])

Labels:
y_train: torch.Size([1024])
y_val  : torch.Size([256])
y_test : torch.Size([320])

Normalization:
Train mean: 0.00017230
Train std : 0.71135187

DATA READY


# LSTM

In [8]:
# ============================================================
# PQD-16c LSTM
# Training + Best Model Saving + Testing
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path


# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
EPOCHS = 100
PATIENCE = 15

NUM_CLASSES = 16
INPUT_LENGTH = 640

MODEL_PATH = Path("../models/legacy/lstm_pq_best.pth")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. PREPARE DATA
# ============================================================
# X shape: [N, 1, 640]
# LSTM needs: [N, 640, 1]

X_train_t = torch.tensor(
    X_train, dtype=torch.float32
).transpose(1, 2)

X_val_t = torch.tensor(
    X_val, dtype=torch.float32
).transpose(1, 2)

X_test_t = torch.tensor(
    X_test, dtype=torch.float32
).transpose(1, 2)

y_train_t = torch.tensor(
    y_train, dtype=torch.long
)

y_val_t = torch.tensor(
    y_val, dtype=torch.long
)

y_test_t = torch.tensor(
    y_test, dtype=torch.long
)


# ============================================================
# 4. DATA LOADERS
# ============================================================

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# 5. LSTM MODEL
# ============================================================

class LSTMModel(nn.Module):

    def __init__(self, num_classes=16):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.25
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        return self.classifier(x[:, -1, :])


# ============================================================
# 6. CREATE MODEL
# ============================================================

model = LSTMModel(NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=7,
    min_lr=1e-6
)


# ============================================================
# 7. TRAINING VARIABLES
# ============================================================

best_loss = float("inf")
best_acc = 0.0
best_epoch = 0
best_state = None
wait = 0


# ============================================================
# 8. TRAINING
# ============================================================

print("\n" + "=" * 70)
print("PQD-16c LSTM TRAINING")
print("=" * 70)

print("Device:", DEVICE)
print("Model :", MODEL_PATH.resolve())
print("=" * 70)


for epoch in range(1, EPOCHS + 1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (
            outputs.argmax(1) == yb
        ).sum().item()
        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            outputs = model(xb)

            loss = criterion(outputs, yb)

            val_loss += loss.item() * xb.size(0)

            val_correct += (
                outputs.argmax(1) == yb
            ).sum().item()

            val_total += yb.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    scheduler.step(val_loss)

    lr = optimizer.param_groups[0]["lr"]


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch:03d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:6.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:6.2f}% | "
        f"LR: {lr:.2e}"
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL IN MEMORY
    # --------------------------------------------------------

    if val_loss < best_loss:

        best_loss = val_loss
        best_acc = val_acc
        best_epoch = epoch
        wait = 0

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    else:
        wait += 1


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if wait >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}"
        )

        break


# ============================================================
# 9. RESTORE BEST MODEL
# ============================================================

model.load_state_dict(best_state)
model.to(DEVICE)


# ============================================================
# 10. SAVE BEST MODEL
# ============================================================

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_classes": NUM_CLASSES,
        "input_length": INPUT_LENGTH,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_acc,
        "best_val_loss": best_loss,
        "architecture": "LSTM"
    },
    MODEL_PATH
)

print("\n" + "=" * 70)
print("BEST LSTM MODEL SAVED")
print("=" * 70)

print("Best Epoch     :", best_epoch)
print(f"Best Val Loss  : {best_loss:.4f}")
print(f"Best Val Acc   : {best_acc:.2f}%")
print("Model Path     :", MODEL_PATH.resolve())


# ============================================================
# 11. TEST
# ============================================================

model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

all_preds = []
all_targets = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        outputs = model(xb)

        loss = criterion(outputs, yb)

        predictions = outputs.argmax(1)

        test_loss += loss.item() * xb.size(0)

        test_correct += (
            predictions == yb
        ).sum().item()

        test_total += yb.size(0)

        all_preds.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            yb.cpu().numpy()
        )


test_loss /= test_total
test_acc = 100 * test_correct / test_total


# ============================================================
# 12. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("LSTM FINAL RESULTS")
print("=" * 70)

print(f"Best Epoch        : {best_epoch}")
print(f"Best Val Loss     : {best_loss:.4f}")
print(f"Best Val Accuracy : {best_acc:.2f}%")
print(f"Test Loss         : {test_loss:.4f}")
print(f"Test Accuracy     : {test_acc:.2f}%")
print(f"Correct           : {test_correct}/{test_total}")
print(f"Wrong             : {test_total - test_correct}/{test_total}")
print(f"Model Saved       : {MODEL_PATH.resolve()}")

print("=" * 70)


PQD-16c LSTM TRAINING
Device: cpu
Model : /mnt/ntfs/my_docs/VIT_Y4_SEM7/C1_XAI/project/models/lstm_pq_best.pth
Epoch [001/100] | Train Loss: 2.7758 | Train Acc:   6.25% | Val Loss: 2.7753 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [002/100] | Train Loss: 2.7754 | Train Acc:   6.54% | Val Loss: 2.7748 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [003/100] | Train Loss: 2.7750 | Train Acc:   6.84% | Val Loss: 2.7740 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [004/100] | Train Loss: 2.7736 | Train Acc:   6.35% | Val Loss: 2.7727 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [005/100] | Train Loss: 2.7709 | Train Acc:   6.45% | Val Loss: 2.7673 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [006/100] | Train Loss: 2.7604 | Train Acc:   7.91% | Val Loss: 2.7357 | Val Acc:   6.64% | LR: 2.00e-04
Epoch [007/100] | Train Loss: 2.7159 | Train Acc:   9.08% | Val Loss: 2.6176 | Val Acc:   9.77% | LR: 2.00e-04
Epoch [008/100] | Train Loss: 2.5891 | Train Acc:  14.06% | Val Loss: 2.5253 | Val Acc:  14.45% | LR: 2.00e-04


# BiLSTM

In [9]:
# ============================================================
# PQD-16c BiLSTM
# Training + Best Model Saving + Testing
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path


# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
EPOCHS = 100
PATIENCE = 15

NUM_CLASSES = 16
INPUT_LENGTH = 640

MODEL_PATH = Path("../models/legacy/bilstm_pq_best.pth")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. PREPARE DATA
# ============================================================
# X shape: [N, 1, 640]
# BiLSTM needs: [N, 640, 1]

X_train_t = torch.tensor(
    X_train, dtype=torch.float32
).transpose(1, 2)

X_val_t = torch.tensor(
    X_val, dtype=torch.float32
).transpose(1, 2)

X_test_t = torch.tensor(
    X_test, dtype=torch.float32
).transpose(1, 2)

y_train_t = torch.tensor(
    y_train, dtype=torch.long
)

y_val_t = torch.tensor(
    y_val, dtype=torch.long
)

y_test_t = torch.tensor(
    y_test, dtype=torch.long
)


# ============================================================
# 4. DATA LOADERS
# ============================================================

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# 5. BiLSTM MODEL
# ============================================================

class BiLSTMModel(nn.Module):

    def __init__(self, num_classes=16):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.25,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        x, _ = self.lstm(x)

        return self.classifier(x[:, -1, :])


# ============================================================
# 6. CREATE MODEL
# ============================================================

model = BiLSTMModel(NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=7,
    min_lr=1e-6
)


# ============================================================
# 7. TRAINING VARIABLES
# ============================================================

best_loss = float("inf")
best_acc = 0.0
best_epoch = 0
best_state = None
wait = 0


# ============================================================
# 8. TRAINING
# ============================================================

print("\n" + "=" * 70)
print("PQD-16c BiLSTM TRAINING")
print("=" * 70)

print("Device:", DEVICE)
print("Model :", MODEL_PATH.resolve())
print("=" * 70)


for epoch in range(1, EPOCHS + 1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss += loss.item() * xb.size(0)

        train_correct += (
            outputs.argmax(1) == yb
        ).sum().item()

        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            outputs = model(xb)

            loss = criterion(outputs, yb)

            val_loss += loss.item() * xb.size(0)

            val_correct += (
                outputs.argmax(1) == yb
            ).sum().item()

            val_total += yb.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    scheduler.step(val_loss)

    lr = optimizer.param_groups[0]["lr"]


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch:03d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:6.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:6.2f}% | "
        f"LR: {lr:.2e}"
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL IN MEMORY
    # --------------------------------------------------------

    if val_loss < best_loss:

        best_loss = val_loss
        best_acc = val_acc
        best_epoch = epoch
        wait = 0

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    else:
        wait += 1


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if wait >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}"
        )

        break


# ============================================================
# 9. RESTORE BEST MODEL
# ============================================================

model.load_state_dict(best_state)
model.to(DEVICE)


# ============================================================
# 10. SAVE BEST MODEL
# ============================================================

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_classes": NUM_CLASSES,
        "input_length": INPUT_LENGTH,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_acc,
        "best_val_loss": best_loss,
        "architecture": "BiLSTM"
    },
    MODEL_PATH
)

print("\n" + "=" * 70)
print("BEST BiLSTM MODEL SAVED")
print("=" * 70)

print("Best Epoch     :", best_epoch)
print(f"Best Val Loss  : {best_loss:.4f}")
print(f"Best Val Acc   : {best_acc:.2f}%")
print("Model Path     :", MODEL_PATH.resolve())


# ============================================================
# 11. TEST
# ============================================================

model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

all_preds = []
all_targets = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        outputs = model(xb)

        loss = criterion(outputs, yb)

        predictions = outputs.argmax(1)

        test_loss += loss.item() * xb.size(0)

        test_correct += (
            predictions == yb
        ).sum().item()

        test_total += yb.size(0)

        all_preds.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            yb.cpu().numpy()
        )


test_loss /= test_total
test_acc = 100 * test_correct / test_total


# ============================================================
# 12. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BiLSTM FINAL RESULTS")
print("=" * 70)

print(f"Best Epoch        : {best_epoch}")
print(f"Best Val Loss     : {best_loss:.4f}")
print(f"Best Val Accuracy : {best_acc:.2f}%")
print(f"Test Loss         : {test_loss:.4f}")
print(f"Test Accuracy     : {test_acc:.2f}%")
print(f"Correct           : {test_correct}/{test_total}")
print(f"Wrong             : {test_total - test_correct}/{test_total}")
print(f"Model Saved       : {MODEL_PATH.resolve()}")

print("=" * 70)


PQD-16c BiLSTM TRAINING
Device: cpu
Model : /mnt/ntfs/my_docs/VIT_Y4_SEM7/C1_XAI/project/models/bilstm_pq_best.pth
Epoch [001/100] | Train Loss: 2.7750 | Train Acc:   6.25% | Val Loss: 2.7744 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [002/100] | Train Loss: 2.7746 | Train Acc:   6.35% | Val Loss: 2.7738 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [003/100] | Train Loss: 2.7741 | Train Acc:   6.54% | Val Loss: 2.7732 | Val Acc:   6.25% | LR: 2.00e-04
Epoch [004/100] | Train Loss: 2.7732 | Train Acc:   8.30% | Val Loss: 2.7723 | Val Acc:   8.20% | LR: 2.00e-04
Epoch [005/100] | Train Loss: 2.7720 | Train Acc:   9.38% | Val Loss: 2.7707 | Val Acc:  12.50% | LR: 2.00e-04
Epoch [006/100] | Train Loss: 2.7690 | Train Acc:  10.16% | Val Loss: 2.7652 | Val Acc:  12.50% | LR: 2.00e-04
Epoch [007/100] | Train Loss: 2.7581 | Train Acc:  11.04% | Val Loss: 2.7340 | Val Acc:  15.62% | LR: 2.00e-04
Epoch [008/100] | Train Loss: 2.7190 | Train Acc:  10.35% | Val Loss: 2.6684 | Val Acc:  10.55% | LR: 2.00e

# Residual 1D CNN

In [3]:
# ============================================================
# PQD-16c ResCNN V1
# Optimized Residual 1D CNN for Power Quality Classification
# ============================================================

import copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader

# -------------------- CONFIG --------------------

SEED = 42
DATA_DIR = Path("../dataset/PQD-16c-Data")
TRAIN_CSV, VAL_CSV, TEST_CSV = [DATA_DIR / f for f in
                                ["train.csv", "validation.csv", "test.csv"]]

BATCH_SIZE = 64
EPOCHS = 100
PATIENCE = 15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("PQD-16c ResCNN V1")
print("=" * 60)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# -------------------- LOAD DATA --------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

assert train_df.columns.tolist() == val_df.columns.tolist() == test_df.columns.tolist()

LABEL_COLUMN = train_df.columns[0]
SIGNAL_COLUMNS = train_df.columns[1:]

assert len(SIGNAL_COLUMNS) == 640

X_train = train_df[SIGNAL_COLUMNS].values.astype(np.float32)
X_val = val_df[SIGNAL_COLUMNS].values.astype(np.float32)
X_test = test_df[SIGNAL_COLUMNS].values.astype(np.float32)

y_train_raw = train_df[LABEL_COLUMN].values
y_val_raw = val_df[LABEL_COLUMN].values
y_test_raw = test_df[LABEL_COLUMN].values

class_names = sorted(np.unique(y_train_raw))
NUM_CLASSES = len(class_names)
label_to_index = {label: i for i, label in enumerate(class_names)}

y_train = np.array([label_to_index[x] for x in y_train_raw], dtype=np.int64)
y_val = np.array([label_to_index[x] for x in y_val_raw], dtype=np.int64)
y_test = np.array([label_to_index[x] for x in y_test_raw], dtype=np.int64)

assert NUM_CLASSES == 16
assert X_train.shape == (1024, 640)
assert X_val.shape == (256, 640)
assert X_test.shape == (320, 640)

print("\nDataset:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

print("\nClasses:")
for i, name in enumerate(class_names):
    print(f"{i:2d} -> {name}")


# -------------------- NORMALIZATION --------------------

train_mean, train_std = X_train.mean(), X_train.std()

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

def make_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32).unsqueeze(1), \
           torch.tensor(y, dtype=torch.long)

X_train_t, y_train_t = make_tensor(X_train, y_train)
X_val_t, y_val_t = make_tensor(X_val, y_val)
X_test_t, y_test_t = make_tensor(X_test, y_test)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)


# -------------------- AUGMENTATION --------------------

def augment_signal(x):
    x = x.clone()

    if random.random() < 0.5:
        x += torch.randn_like(x) * 0.008

    if random.random() < 0.4:
        scale = torch.empty(x.size(0), 1, 1, device=x.device).uniform_(0.97, 1.03)
        x *= scale

    if random.random() < 0.2:
        shift = torch.empty(x.size(0), 1, 1, device=x.device).uniform_(-0.01, 0.01)
        x += shift

    return x


# -------------------- RESIDUAL BLOCK --------------------

class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1, dropout=0.05):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels, out_channels, 5, stride, 2, bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels, out_channels, 5, padding=2, bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)

        self.shortcut = (
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
            if stride != 1 or in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        identity = self.shortcut(x)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))

        return self.relu(x + identity)


# -------------------- MODEL --------------------

class CNNPQ(nn.Module):

    def __init__(self, num_classes=16):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True)
        )

        self.features = nn.Sequential(
            ResidualBlock(32, 32, stride=2, dropout=0.05),
            ResidualBlock(32, 64, stride=2, dropout=0.05),
            ResidualBlock(64, 96, stride=2, dropout=0.10),
            ResidualBlock(96, 96, stride=1, dropout=0.10)
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(96, 48),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(48, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


# -------------------- TRAINING SETUP --------------------

model = CNNPQ(NUM_CLASSES).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 60)
print("MODEL")
print("=" * 60)
print(model)
print(f"\nTrainable parameters: {num_params:,}")

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=7, min_lr=1e-6
)

best_val_loss = float("inf")
best_val_acc = 0.0
best_state = None
epochs_without_improvement = 0


# -------------------- TRAINING --------------------

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

for epoch in range(EPOCHS):

    # Train
    model.train()
    train_loss = train_correct = train_total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(augment_signal(xb))
        loss = criterion(logits, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (logits.argmax(1) == yb).sum().item()
        train_total += xb.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # Validation
    model.eval()
    val_loss = val_correct = val_total = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_total += xb.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    scheduler.step(val_loss)

    lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1:03d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:6.2f}% | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:6.2f}% | "
        f"LR: {lr:.2e}"
    )

    # Best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break


# -------------------- RESTORE BEST --------------------

if best_state is not None:
    model.load_state_dict(best_state)

print("\n" + "=" * 60)
print("BEST VALIDATION RESULT")
print("=" * 60)
print(f"Best Val Loss : {best_val_loss:.4f}")
print(f"Best Val Acc  : {best_val_acc:.2f}%")


# -------------------- TEST --------------------

model.eval()
test_loss = test_correct = test_total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        test_loss += loss.item() * xb.size(0)
        test_correct += (logits.argmax(1) == yb).sum().item()
        test_total += xb.size(0)

test_loss /= test_total
test_acc = 100 * test_correct / test_total


# -------------------- SAVE --------------------

MODEL_PATH = Path("../models/legacy/cnn_pq_best.pth")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "num_classes": NUM_CLASSES,
    "class_names": class_names,
    "input_shape": [1, 640],
    "train_mean": float(train_mean),
    "train_std": float(train_std),
    "test_accuracy": test_acc,
    "best_val_accuracy": best_val_acc,
    "best_val_loss": best_val_loss
}, MODEL_PATH)


# -------------------- FINAL RESULT --------------------

print("\n" + "=" * 60)
print("PQD-16c ResCNN V1 RESULTS")
print("=" * 60)
print(f"Best Val Loss : {best_val_loss:.4f}")
print(f"Best Val Acc  : {best_val_acc:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Acc      : {test_acc:.2f}%")
print(f"Model saved   : {MODEL_PATH.resolve()}")
print("=" * 60)

PQD-16c ResCNN V1
Device: cpu

Dataset:
Train: (1024, 640) (1024,)
Val  : (256, 640) (256,)
Test : (320, 640) (320,)

Classes:
 0 -> Flicker
 1 -> Flicker+Harmonics
 2 -> Flicker+Sag
 3 -> Flicker+Swell
 4 -> Harmonics
 5 -> Impulsive Transient
 6 -> Interruption
 7 -> Interruption+Harmonics
 8 -> Normal
 9 -> Notch
10 -> Oscillatory transient
11 -> Sag
12 -> Sag+Harmonics
13 -> Spike
14 -> Swell
15 -> Swell+Harmonics

MODEL
CNNPQ(
  (stem): Sequential(
    (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (features): Sequential(
    (0): ResidualBlock(
      (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
  

# CNN-Res-BiLSTM V2

## CNN + Residual Convolutional Blocks + Dilated Convolutions + BiLSTM + Attention

In [5]:
# ============================================================
# PQD-16c CNN-RES-BiLSTM V2
# ============================================================

import os, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 150
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 25
LR_PATIENCE = 8
LR_FACTOR = 0.5
MIN_LR = 1e-6
NUM_CLASSES = 16
INPUT_LENGTH = 640

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, "cnn_res_bilstm_pq.pth")

print("=" * 70)
print("PQD-16c CNN-RES-BiLSTM V2")
print("=" * 70)
print("Device     :", DEVICE)
print("Project    :", PROJECT_ROOT)
print("Model path :", MODEL_PATH)
print("=" * 70)


# Data shape
def fix_shape(X):
    X = np.asarray(X)

    if X.ndim == 4:
        if X.shape[2] != 1:
            raise ValueError(f"Unexpected 4D input shape: {X.shape}")
        X = np.squeeze(X, axis=2)
    elif X.ndim == 2:
        X = X[:, None, :]
    elif X.ndim != 3:
        raise ValueError(f"Unsupported input shape: {X.shape}")

    if X.shape[1] != 1 or X.shape[2] != INPUT_LENGTH:
        raise ValueError(f"Expected (N, 1, {INPUT_LENGTH}), got {X.shape}")

    return X


# NOTE:
# X_train, y_train, X_val, y_val, X_test, y_test
# must already be loaded by your dataset cell.

X_train, X_val, X_test = map(fix_shape, (X_train, X_val, X_test))
y_train, y_val, y_test = map(
    lambda y: np.asarray(y).reshape(-1),
    (y_train, y_val, y_test)
)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

print("\nDataset shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

class_counts = np.bincount(y_train, minlength=NUM_CLASSES)
print("\nClass counts:", class_counts)
print(
    "Dataset is perfectly balanced."
    if np.all(class_counts == class_counts[0])
    else "WARNING: Dataset is not perfectly balanced."
)

# DataLoaders
def make_loader(X, y, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

train_loader = make_loader(X_train, y_train, True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)


# Residual CNN block
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, dilation=1):
        super().__init__()
        padding = ((kernel_size - 1) * dilation) // 2

        self.conv1 = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation, bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.act1 = nn.GELU()

        self.conv2 = nn.Conv1d(
            out_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation, bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.skip = (
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm1d(out_channels)
            )
            if in_channels != out_channels else nn.Identity()
        )

        self.activation = nn.GELU()

    def forward(self, x):
        identity = self.skip(x)
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.activation(x + identity)


# CNN-RES-BiLSTM V2
class CNNResBiLSTM(nn.Module):
    def __init__(self, num_classes=16):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32),
            nn.GELU(),

            ResidualConvBlock(32, 32, 5, 1),
            nn.MaxPool1d(2),

            ResidualConvBlock(32, 64, 5, 1),
            nn.MaxPool1d(2),

            ResidualConvBlock(64, 128, 5, 2),
            nn.MaxPool1d(2),

            ResidualConvBlock(128, 128, 3, 2)
        )

        self.sequence_norm = nn.LayerNorm(128)

        self.lstm = nn.LSTM(
            128, 128,
            num_layers=2,
            batch_first=True,
            dropout=0.25,
            bidirectional=True
        )

        self.attention = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if x.ndim == 4:
            x = x.squeeze(2)

        x = self.cnn(x).transpose(1, 2)
        x = self.sequence_norm(x)
        x, _ = self.lstm(x)

        weights = torch.softmax(self.attention(x), dim=1)
        x = torch.sum(x * weights, dim=1)

        return self.classifier(x)


# Model
model = CNNResBiLSTM(NUM_CLASSES).to(DEVICE)

num_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("\n" + "=" * 70)
print(model)
print("=" * 70)
print(f"Trainable parameters: {num_parameters:,}")
print("=" * 70)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=MIN_LR
)

best_val_acc = 0.0
best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0


# Training
for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = train_correct = train_total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (outputs.argmax(1) == yb).sum().item()
        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # Validation
    model.eval()
    val_loss = val_correct = val_total = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            outputs = model(xb)
            loss = criterion(outputs, yb)

            val_loss += loss.item() * xb.size(0)
            val_correct += (outputs.argmax(1) == yb).sum().item()
            val_total += yb.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch:03d}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:6.2f}% "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:6.2f}% "
        f"| LR: {current_lr:.2e}"
    )

    is_better = (
        val_acc > best_val_acc or
        (val_acc == best_val_acc and val_loss < best_val_loss)
    )

    if is_better:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "num_classes": NUM_CLASSES,
            "input_length": INPUT_LENGTH,
            "best_val_accuracy": best_val_acc,
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "architecture": "CNNResBiLSTM"
        }, MODEL_PATH)

        print(f"  >>> BEST MODEL SAVED ({best_val_acc:.2f}%)")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break


# Load best model
print("\n" + "=" * 70)
print("LOADING BEST MODEL")
print("=" * 70)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Best epoch: {checkpoint['best_epoch']}")
print(f"Best validation accuracy: {checkpoint['best_val_accuracy']:.2f}%")
print(f"Model loaded from:\n{MODEL_PATH}")


# Test
test_loss = test_correct = test_total = 0
all_preds, all_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        outputs = model(xb)
        loss = criterion(outputs, yb)
        predictions = outputs.argmax(1)

        test_loss += loss.item() * xb.size(0)
        test_correct += (predictions == yb).sum().item()
        test_total += yb.size(0)

        all_preds.extend(predictions.cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

test_loss /= test_total
test_acc = 100 * test_correct / test_total

print("\n" + "=" * 70)
print("CNN-RES-BiLSTM V2 RESULTS")
print("=" * 70)
print(f"Best Epoch        : {best_epoch}")
print(f"Best Val Loss     : {best_val_loss:.4f}")
print(f"Best Val Accuracy : {best_val_acc:.2f}%")
print(f"Test Loss         : {test_loss:.4f}")
print(f"Test Accuracy     : {test_acc:.2f}%")
print(f"Correct           : {test_correct}/{test_total}")
print(f"Wrong             : {test_total - test_correct}/{test_total}")
print(f"Model saved       : {MODEL_PATH}")
print("=" * 70)

PQD-16c CNN-RES-BiLSTM V2
Device     : cpu
Project    : /mnt/ntfs/my_docs/VIT_Y4_SEM7/C1_XAI/project
Model path : /mnt/ntfs/my_docs/VIT_Y4_SEM7/C1_XAI/project/models/cnn_res_bilstm_pq.pth

Dataset shapes:
Train: (1024, 1, 640) (1024,)
Val  : (256, 1, 640) (256,)
Test : (320, 1, 640) (320,)

Class counts: [64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64]
Dataset is perfectly balanced.

CNNResBiLSTM(
  (cnn): Sequential(
    (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): ResidualConvBlock(
      (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act1): GELU(approximate='none')
      (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (bn2): Batch

# End